# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date published: {metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, columns, and their corresponding `@id` values for exploration.

In [ ]:
# List all available RecordSets and their details using mlcroissant.
print("List of record sets and fields:")

# `record_sets` is a list of mlcroissant.RecordSet objects.
record_sets = list(dataset.metadata.record_sets)

for rs in record_sets:
    print(f"\nRecord set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set, field, and column `@id`s from the overview for selection.

In [ ]:
# Extract data from each record set.
# We'll build a dictionary where key=@id of the record set, value=DataFrame for the records
dataframes = {}

# Collect list of all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

print(f"Record sets to load: {record_set_ids}")

# Load data for each record set
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records_gen = dataset.records(record_set=record_set_id)
    rows = list(records_gen)
    if rows:
        df = pd.DataFrame(rows)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}")

# Display available columns for each record set
for rec_id, df in dataframes.items():
    print(f"\nColumns for record set {rec_id}: \n{df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing, and grouping by categorical fields.

**Note:** Fill in the correct field `@id`s for your analysis based on the overview step.

In [ ]:
# For demonstration, select the main protein data table record set and fields by their @id
from pprint import pprint

# Select main data table (fill in record set id if only one exists)
main_record_set_id = record_set_ids[0]  # Replace with actual @id if more than one
df = dataframes[main_record_set_id]

print(f"Analyzing columns for record set {main_record_set_id}:")
pprint(df.columns.tolist())

# Try to pick a numeric field and a group field, usually these have meaningful names like 'coverage'/'MW'/'Abundance...'
# Example field ids: 'coverage', 'mw', 'abundance', etc.
# For demonstration, pick the first numeric-looking field
import numpy as np

numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) or np.issubdtype(df[c].dtype, np.number)]
if not numeric_candidates:
    # Try to convert object columns to numeric if possible
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
        except Exception:
            continue
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) or np.issubdtype(df[c].dtype, np.number)]
print(f"Numeric field candidates: {numeric_candidates}")

# Use the first numeric field for filtering/normalizing if possible
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df.loc[:, f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a group field (categorical non-numeric)
    group_candidates = [c for c in df.columns if (df[c].dtype == 'object' and df[c].nunique() > 1 and df[c].nunique() < 30)]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"\nGrouping by {group_field}...")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we display a histogram of the main numeric field and a bar plot for grouped values if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Bar plot of mean value by group if available
if 'grouped_df' in locals() and 'group_field' in locals():
    plt.figure(figsize=(10, 6))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Average {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

- We loaded dataset metadata and reviewed available record sets and fields by their `@id`.
- We extracted and inspected records, performed exploratory data analysis by filtering and normalizing numeric fields, and demonstrated grouping by categorical attributes.
- Visualizations were used to summarize data distributions and group means.

This reproducible workflow can be extended for more advanced analyses and adapted to more complex Croissant-formatted datasets.